In [1]:
import torch
print("Có GPU không? :", torch.cuda.is_available())  # Phải trả về True
print("Phiên bản CUDA trong PyTorch:", torch.version.cuda)
print("Tên GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Không có GPU")

# Kiểm tra GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang sử dụng: {device}") # phải là: Đang sử dụng: cuda

Có GPU không? : True
Phiên bản CUDA trong PyTorch: 11.8
Tên GPU: NVIDIA GeForce GTX 1650
Đang sử dụng: cuda


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
from PIL import Image
import numpy as np
import os

In [3]:
# Kiểm tra GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang sử dụng: {device}")

Đang sử dụng: cuda


In [4]:
# Load mô hình ResNet50 đã pretrain
base_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
num_ftrs = base_model.fc.in_features

In [6]:
# Thêm các lớp fully connected (số nhãn của dataset)
num_classes = 8
base_model.fc = nn.Sequential(
    nn.Linear(num_ftrs, 1024),
    nn.ReLU(),
    nn.Linear(1024, num_classes),
    nn.Softmax(dim=1)
)

In [7]:
# Chuyển model sang GPU
base_model = base_model.to(device)

In [8]:
# Đóng băng các layer của ResNet50
for param in base_model.parameters():
    param.requires_grad = False

In [9]:
# Chỉ fine-tune 4 lớp cuối
for param in list(base_model.parameters())[-4:]:
    param.requires_grad = True

In [10]:
# Optimizer và loss function
optimizer = optim.Adam(base_model.fc.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [14]:
# Chuẩn bị dữ liệu
train_dir = "data/images/train"
val_dir = "data/images/valid"
test_dir = "data/images/test"

In [15]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [16]:
train_dataset = datasets.ImageFolder(train_dir, transform=transform)
val_dataset = datasets.ImageFolder(val_dir, transform=transform)
test_dataset = datasets.ImageFolder(test_dir, transform=transform)


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [17]:
# Training loop
def train(model, train_loader, val_loader, epochs=10):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
        
        acc = 100 * correct / total
        print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}, Accuracy: {acc:.2f}%")

In [18]:
# Huấn luyện mô hình
train(base_model, train_loader, val_loader, epochs=10)


Epoch 1, Loss: 2.0664, Accuracy: 18.32%
Epoch 2, Loss: 1.9910, Accuracy: 27.81%
Epoch 3, Loss: 1.9272, Accuracy: 34.66%
Epoch 4, Loss: 1.8983, Accuracy: 37.75%
Epoch 5, Loss: 1.8037, Accuracy: 49.67%
Epoch 6, Loss: 1.7138, Accuracy: 60.04%
Epoch 7, Loss: 1.6686, Accuracy: 63.58%
Epoch 8, Loss: 1.6574, Accuracy: 65.12%
Epoch 9, Loss: 1.6227, Accuracy: 65.78%
Epoch 10, Loss: 1.6136, Accuracy: 65.78%


In [19]:
# Fine-tune 4 lớp cuối
train(base_model, train_loader, val_loader, epochs=2)

Epoch 1, Loss: 1.5233, Accuracy: 77.26%
Epoch 2, Loss: 1.5037, Accuracy: 78.59%


In [20]:
# Lưu mô hình
torch.save(base_model.state_dict(), "fine_tuned_model.pth")

In [21]:
# Load mô hình
tuned_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
tuned_model.fc = base_model.fc  # Gán lại phần fully connected

In [22]:
tuned_model.load_state_dict(torch.load("fine_tuned_model.pth"))
tuned_model = tuned_model.to(device)
tuned_model.eval()


C:\Users\NewTun\AppData\Local\Temp\ipykernel_16728\2224363281.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  tuned_model.load_state_dict(torch.load("fine_tuned_model.pt

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [24]:
# Dự đoán trên một ảnh đơn
img_path = "data/images/test/Portmap/2.png"
img = Image.open(img_path)
img = transform(img).unsqueeze(0).to(device)

with torch.no_grad():
    output = tuned_model(img)
    predicted_class = torch.argmax(output, dim=1).item()
    print(f"Dự đoán nhãn: {predicted_class}")

Dự đoán nhãn: 0


In [25]:
# Đánh giá trên tập test
def evaluate(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    test_loss = 0.0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    
    acc = 100 * correct / total
    print(f"Test Loss: {test_loss/len(test_loader):.4f}")
    print(f"Test Accuracy: {acc:.2f}%")

evaluate(tuned_model, test_loader)

Test Loss: 1.5460
Test Accuracy: 75.50%
